In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 5.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, f1_score
from imblearn.over_sampling import SMOTE
import optuna
from optuna.samplers import TPESampler
import joblib
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline

In [3]:
df = pd.read_csv("/content/cancer-risk-factors.csv")
df.head(5)

,Patient_ID,Cancer_Type,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,...,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level,Risk_Level
0,LU0000,Breast,68,0,7,2,8,0,5,3,...,4,6,3,1,0,0,0.398696,28.0,5,Medium
1,LU0001,Prostate,74,1,8,9,8,0,0,3,...,1,3,3,0,0,5,0.424299,25.4,9,Medium
2,LU0002,Skin,55,1,7,10,7,0,3,3,...,1,8,10,0,0,6,0.605082,28.6,2,Medium
3,LU0003,Colon,61,0,6,2,2,0,6,2,...,6,4,8,0,0,8,0.318449,32.1,7,Low
4,LU0004,Lung,67,1,10,7,4,0,6,3,...,9,10,9,0,0,5,0.524358,25.1,2,Medium


In [4]:
X =  df.drop(columns=["Patient_ID", "Risk_Level", "Cancer_Type"])

#encode the target variable
le= LabelEncoder()
y = le.fit_transform(df["Risk_Level"])

In [9]:
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [ ]:
features_name = X.columns
features_name

Index(['Age', 'Gender', 'Smoking', 'Alcohol_Use', 'Obesity', 'Family_History',
       'Diet_Red_Meat', 'Diet_Salted_Processed', 'Fruit_Veg_Intake',
       'Physical_Activity', 'Air_Pollution', 'Occupational_Hazards',
       'BRCA_Mutation', 'H_Pylori_Infection', 'Calcium_Intake',
       'Overall_Risk_Score', 'BMI', 'Physical_Activity_Level'],
      dtype='object')

In [ ]:
X.head(5)

,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,Fruit_Veg_Intake,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level
0,68,0,7,2,8,0,5,3,7,4,6,3,1,0,0,0.398696,28.0,5
1,74,1,8,9,8,0,0,3,7,1,3,3,0,0,5,0.424299,25.4,9
2,55,1,7,10,7,0,3,3,4,1,8,10,0,0,6,0.605082,28.6,2
3,61,0,6,2,2,0,6,2,4,6,4,8,0,0,8,0.318449,32.1,7
4,67,1,10,7,4,0,6,3,10,9,10,9,0,0,5,0.524358,25.1,2


In [ ]:
y

array([2, 2, 2, ..., 1, 2, 2])

In [5]:
#split into train and test

X_train,X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y) #stratify - ensures each split keeps the same proportion of classes

In [6]:
y_train

array([2, 2, 2, ..., 2, 2, 2])

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred = model.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix Report")
print(confusion_matrix(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.95      0.97        20
         Low       1.00      1.00      1.00        65
      Medium       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       1.00      0.98      0.99       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix Report
[[ 19   0   1]
 [  0  65   0]
 [  0   0 315]]


The modelaccurately distinguishes all risk level, with only 1 mistake

Logistic Regression Model

In [ ]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
y_pred_lr = log_reg.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

print("Confusion Matrix Report")
print(confusion_matrix(y_test, y_pred_lr))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.80      0.89        20
         Low       0.97      0.95      0.96        65
      Medium       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix Report
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


Removing Overall Risk Score

In [ ]:
X_new =  df.drop(columns=["Patient_ID", "Risk_Level", "Cancer_Type", "Overall_Risk_Score" ])

#encode the target variable
le_new= LabelEncoder()
y_new = le_new.fit_transform(df["Risk_Level"])


In [ ]:
features_names = X_new.columns
joblib.dump(features_name,"features_names.pkl")

['features_names.pkl']

In [ ]:
#split into train and test

X_train_new ,X_test_new , y_train_new , y_test_new = train_test_split(X_new,y_new, test_size=0.2, random_state=42, stratify=y) #stratify - ensures each split keeps the same proportion of classes

In [ ]:
# Feature Scaling
scaler_new = StandardScaler()
X_train_new = scaler_new.fit_transform(X_train)
X_test_new = scaler_new.transform(X_test)

In [ ]:
model_new = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train_new, y_train_new)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred_new = model.predict(X_test_new)

print("Classification Report")
print(classification_report(y_test_new, y_pred_new, target_names=le.classes_))

print("Confusion Matrix Report")
print(confusion_matrix(y_test_new, y_pred_new))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.95      0.97        20
         Low       1.00      1.00      1.00        65
      Medium       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       1.00      0.98      0.99       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix Report
[[ 19   0   1]
 [  0  65   0]
 [  0   0 315]]


In [ ]:
log_reg_new = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_new,y_train_new)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
y_pred_lr_new = log_reg.predict(X_test_new)

print("Classification Report")
print(classification_report(y_test_new, y_pred_lr_new, target_names=le.classes_))

print("Confusion Matrix Report")
print(confusion_matrix(y_test_new, y_pred_lr_new))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.80      0.89        20
         Low       0.97      0.95      0.96        65
      Medium       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix Report
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


Balancing the Data to get better result.

## SMOTE - Synthetic Minority Over Sampling Technique

In [ ]:
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_new, y_train_new)

print("Class Distribution after SMOTE")
print(pd.Series(y_train_res).value_counts())

Class Distribution after SMOTE
2    1259
1    1259
0    1259
Name: count, dtype: int64


In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_res, y_train_res)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred_rf = rf_model.predict(X_test_new)

print("Classification Report")
print(classification_report(y_test_new, y_pred_rf, target_names=le_new.classes_))

print("Confusion Matrix Report")
print(confusion_matrix(y_test_new, y_pred_rf))

Classification Report
              precision    recall  f1-score   support

        High       1.00      1.00      1.00        20
         Low       0.98      1.00      0.99        65
      Medium       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       0.99      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix Report
[[ 20   0   0]
 [  0  65   0]
 [  0   1 314]]


The model is performing well.

## Optuna Tunning


In [ ]:
#define macro F1 scorer explicitily for multiclass
def macro_f1(y_true, y_pred):
  return f1_score(y_true, y_pred, average="macro")

scorer = make_scorer(macro_f1)

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train_res, y_train_res):
        X_t, X_v = X_train_res[train_idx], X_train_res[val_idx]
        y_t, y_v = y_train_res[train_idx], y_train_res[val_idx]

        model.fit(X_t, y_t)
        y_pred = model.predict(X_v)

        score = f1_score(y_v, y_pred, average='macro')  # explicit multiclass handling
        scores.append(score)

    return np.mean(scores)

In [ ]:
# Create Study

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler, study_name="rf_macro_f1")

[I 2026-03-02 17:57:33,748] A new study created in memory with name: rf_macro_f1


In [ ]:
# --- Run optimization ---
n_trials = 50  # change to 100+ if you want more thorough search
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("Best trial:")
print("  Value (macro F1):", study.best_value)
print("  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-02 17:57:44,522] Trial 0 finished with value: 0.9997354493605197 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9997354493605197.
[I 2026-03-02 17:57:46,956] Trial 1 finished with value: 0.9997354493605197 and parameters: {'n_estimators': 59, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.9997354493605197.
[I 2026-03-02 17:57:59,118] Trial 2 finished with value: 0.9997354493605197 and parameters: {'n_estimators': 325, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.9997354493605197.
[I 2026-03-02 17:58:09,793] Trial 3 finished with value: 0.9997354493605197 and parameters: {'n_estimators': 324, 'max_dept

In [ ]:
#Train the model with best parameters and full training data, then evaluate on test

best_params = {
        'n_estimators': 218,
        'max_depth': 29,
        'min_samples_split': 8,
        'min_samples_leaf': 6,
        'max_features': "sqrt",
        'bootstrap': False,
        'criterion': "entropy",
        'random_state': 42,
        'n_jobs': -1
    }

final_rf = RandomForestClassifier(**best_params)
final_rf.fit(X_train_res, y_train_res)
y_pred_test = final_rf.predict(X_test_new)

print(classification_report(y_test_new, y_pred_test))
print(confusion_matrix(y_test_new, y_pred_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.98      1.00      0.99        65
           2       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       0.99      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

[[ 20   0   0]
 [  0  65   0]
 [  0   1 314]]


## XGBoost

In [ ]:
le = LabelEncoder()
le = le.fit(y_train)
joblib.dump(le, "label_encoder.pkl")

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# Build Pipeline
xgb = XGBClassifier(use_label_encoder=False, eval_metrics='mlogloss', random_state=42, n_jobs=-1)
pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', xgb)
])

pipe.fit(X_train, y_train_enc)

y_pred_enc = pipe.predict(X_test)

y_pred = le.inverse_transform(y_pred_enc)
y_test_orig = le.inverse_transform(y_test_enc)

print("Baseline XGB Classifier")
print(classification_report(y_test_orig, y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test_orig, y_pred))

Baseline XGB Classifier
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.97      1.00      0.98        65
           2       1.00      0.99      1.00       315

    accuracy                           0.99       400
   macro avg       0.99      1.00      0.99       400
weighted avg       1.00      0.99      1.00       400

Confusion Matrix
[[ 20   0   0]
 [  0  65   0]
 [  0   2 313]]


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:17:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Both RF And XGBoost Model is giving almost the same result. I will be considering RF model as the model made only 1 mistake.

Saving the model


In [ ]:
joblib.dump(final_rf, "model_rf.pkl")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
## Optuna + Random Forest pipeline that optimizes recall for the High class.

# --- Imports ---
import optuna

from optuna.samplers import TPESampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# --- Settings ---
TARGET_LABEL = 'High'       # label whose recall we want to maximize
LABEL_ORDER = ['High', 'Low', 'Medium']  # must match your label names exactly
N_TRIALS = 40               # start with 40; increase to 100+ for more thorough search
CV_FOLDS = 3

# --- Encode labels once (keep mapping stable) ---
le = LabelEncoder()
# Fit the LabelEncoder on the unique string values from the original DataFrame column
le.fit(df["Risk_Level"].unique())

# y_train and y_test are already numerically encoded using a consistent LabelEncoder
# So we can assign them directly for consistency within this block.
y_train_enc = y_train
y_test_enc  = y_test

# numeric index of target label in encoded space
target_index = int(np.where(le.classes_ == TARGET_LABEL)[0][0]) # This line is now safe

# --- Objective: maximize recall for TARGET_LABEL ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42,
        'n_jobs': 1 # For CV, avoid nested parallelism
    }

    # Build pipeline (SMOTE on training folds only)
    rf = RandomForestClassifier(**params)
    pipe = ImbPipeline([('smote', SMOTE(random_state=42)), ('rf', rf)])

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    recalls = []

    for train_idx, val_idx in cv.split(X_train, y_train_enc):
        X_t, X_v = X_train[train_idx], X_train[val_idx]
        y_t, y_v = y_train_enc[train_idx], y_train_enc[val_idx]

        pipe.fit(X_t, y_t)
        y_pred = pipe.predict(X_v)

        # compute recall per class in LABEL_ORDER and select target_index
        recs = recall_score(y_v, y_pred, labels=list(range(len(le.classes_))), average=None, zero_division=0)
        recall_high = recs[target_index]
        recalls.append(recall_high)

    return float(np.mean(recalls))

# --- Run Optuna study ---
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='rf_high_recall')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nBest CV mean High-recall:", study.best_value)
print("Best params:")
for k,v in study.best_params.items():
    print(f"  {k}: {v}")

# --- Train final pipeline with best params and evaluate on test set ---
best_params = study.best_params.copy()
# ensure required fixed args for final fit
best_params.update({'random_state': 42})
final_rf = RandomForestClassifier(**best_params, n_jobs=-1)

final_pipe = ImbPipeline([('smote', SMOTE(random_state=42)), ('rf', final_rf)])
# fit with encoded y (pipeline will apply SMOTE inside)
final_pipe.fit(X_train, y_train_enc)

# predict (encoded), then decode for readable report
y_test_pred_enc = final_pipe.predict(X_test)
y_test_pred = le.inverse_transform(y_test_pred_enc)
y_test_orig = le.inverse_transform(y_test_enc) # y_test should be decoded, not directly used.

print("\nFINAL Test Classification Report (labels in original names):")
print(classification_report(y_test_orig, y_test_pred, labels=le.classes_))
print("Confusion Matrix (rows=true, cols=predicted, order = le.classes_):")
print(confusion_matrix(y_test_orig, y_test_pred, labels=le.classes_))


IndexError: index 0 is out of bounds for axis 0 with size 0